# 01 — Build NRC Lexicon Lookup Dicts

Loads three NRC lexicons, builds word-level lookup dicts, and pickles them to `data/lexicons/`
so the scoring notebook doesn't have to reload and rejoin on every run.

Run this once (or whenever you update the lexicon files).

In [ ]:
import pickle
import re
from pathlib import Path

import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

LEXICON_DIR = Path('data/lexicons')
LEXICON_DIR.mkdir(parents=True, exist_ok=True)

EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
VAD_DIMS  = ['valence', 'arousal', 'dominance']

_lemmatizer = WordNetLemmatizer()


def _tokenize(keyword: str) -> list:
    """Split phrase on whitespace/hyphens/underscores, lowercase, alpha tokens only."""
    return [t for t in re.split(r'[\s\-_/]+', keyword.lower()) if t.isalpha()]


def _lemma_candidates(tok: str) -> list:
    """Return lemma forms for noun, verb, adjective POS (deduplicated)."""
    seen, out = set(), []
    for pos in ('n', 'v', 'a'):
        lemma = _lemmatizer.lemmatize(tok, pos=pos)
        if lemma not in seen:
            seen.add(lemma)
            out.append(lemma)
    if tok not in seen:
        out.append(tok)
    return out

In [ ]:
# ── NRC EmoLex (binary 0/1 per emotion + positive/negative) ──────────────
NRC_EMOLEX = Path('data/NRC-emotion-lexicon-wordlevel-alphabetized-v0.92.txt')
nrc_long = pd.read_csv(NRC_EMOLEX, sep='\t', header=None,
                       names=['word', 'emotion', 'value'], skiprows=46)
nrc_wide = nrc_long.pivot(index='word', columns='emotion', values='value').reset_index()
print(f'EmoLex: {len(nrc_wide):,} words')

In [ ]:
# ── NRC Emotion Intensity (continuous 0–1 per emotion) ───────────────────
INTENSITY_FILE = Path('data/NRC-Emotion-Intensity-Lexicon/NRC-Emotion-Intensity-Lexicon-v1.txt')
nrc_int_long = pd.read_csv(INTENSITY_FILE, sep='\t', header=None,
                           names=['word', 'emotion', 'intensity'])
nrc_int_wide = (
    nrc_int_long
    .pivot(index='word', columns='emotion', values='intensity')
    .fillna(0)
    .reset_index()
)
print(f'Intensity lexicon: {len(nrc_int_wide):,} words')

In [ ]:
# ── NRC VAD (valence/arousal/dominance 0–1) ──────────────────────────────
VAD_FILE = Path('data/NRC-VAD-Lexicon/NRC-VAD-Lexicon/NRC-VAD-Lexicon.txt')
nrc_vad = pd.read_csv(VAD_FILE, sep='\t', header=None,
                      names=['word', 'valence', 'arousal', 'dominance'])
print(f'VAD lexicon: {len(nrc_vad):,} words')

In [ ]:
# ── Build fast lookup dicts ───────────────────────────────────────────────
intensity_lookup = {
    row['word']: {e: row[e] for e in EMOTIONS}
    for _, row in nrc_int_wide.iterrows()
}

vad_lookup = {
    row['word']: {d: row[d] for d in VAD_DIMS}
    for _, row in nrc_vad.iterrows()
}

print(f'intensity_lookup: {len(intensity_lookup):,} entries')
print(f'vad_lookup:       {len(vad_lookup):,} entries')

In [ ]:
# ── Pickle everything needed by 02_score_and_export ──────────────────────
payload = {
    'intensity_lookup': intensity_lookup,
    'vad_lookup':       vad_lookup,
    'EMOTIONS':         EMOTIONS,
    'VAD_DIMS':         VAD_DIMS,
}

out = LEXICON_DIR / 'nrc_lookups.pkl'
with open(out, 'wb') as f:
    pickle.dump(payload, f, protocol=5)

print(f'Saved → {out}  ({out.stat().st_size / 1e6:.1f} MB)')